In [109]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [110]:
df=pd.read_csv("Data/tickets_dni.csv")
df.head()

,dia,num_ticket,cod_est,id_cliente_enc
0,20251027,17977,12650101,55391756M
1,20251027,17910,8452015,11971559J
2,20251027,17721,8452015,22454313B
3,20251027,17873,8452020,69353513A
4,20251027,17732,12650101,50171830K


In [111]:
df.shape

(5136313, 4)

In [112]:
df.dtypes

dia               int64
num_ticket        int64
cod_est           int64
id_cliente_enc      str
dtype: object

# Analisis exploratorio

In [113]:
df.describe()

,dia,num_ticket,cod_est
count,5.136313e+06,5.136313e+06,5.136313e+06
mean,2.025097e+07,1.469170e+04,7.138375e+06
std,1.092743e+02,2.960831e+03,3.998262e+06
min,2.025080e+07,1.000000e+00,1.012305e+06
25%,2.025090e+07,1.197100e+04,4.032065e+06
50%,2.025100e+07,1.458100e+04,8.051530e+06
75%,2.025103e+07,1.723100e+04,1.022070e+07
max,2.025113e+07,2.824800e+04,1.435350e+07


In [114]:
df.isna().sum()

dia               0
num_ticket        0
cod_est           0
id_cliente_enc    0
dtype: int64

In [115]:
df["num_ticket"].max()  # ver max y mins para ver si hay outliers, aunque no es un numero continuo, es un numero de ticket, pero si hay un numero de ticket muy alto puede ser un error

np.int64(28248)

In [116]:
df["num_ticket"].min()

np.int64(1)

In [117]:
df["dia"]=pd.to_datetime(df["dia"], format="%Y%m%d")
df.head()

,dia,num_ticket,cod_est,id_cliente_enc
0,2025-10-27,17977,12650101,55391756M
1,2025-10-27,17910,8452015,11971559J
2,2025-10-27,17721,8452015,22454313B
3,2025-10-27,17873,8452020,69353513A
4,2025-10-27,17732,12650101,50171830K


Juntar dfs para saber que producto es cada codigo

In [118]:
cod=pd.read_csv("Data/maestroestr.csv")
cod.head()

junto=df.merge(cod, on="cod_est")
junto.head()

,dia,num_ticket,cod_est,id_cliente_enc,descripcion
0,2025-10-27,17977,12650101,55391756M,P.H. SECO BASICO 2C NORMAL
1,2025-10-27,17910,8452015,11971559J,FRUTOS SEC. S/CASCARA NUEZ
2,2025-10-27,17721,8452015,22454313B,FRUTOS SEC. S/CASCARA NUEZ
3,2025-10-27,17873,8452020,69353513A,FRUTOS SEC. S/CASCARA COCKTAIL
4,2025-10-27,17732,12650101,50171830K,P.H. SECO BASICO 2C NORMAL


# Clave única

In [119]:
junto['clave unica'] = junto[['dia', 'num_ticket', 'id_cliente_enc']].astype(str).sum(axis=1)
junto.head()

,dia,num_ticket,cod_est,id_cliente_enc,descripcion,clave unica
0,2025-10-27,17977,12650101,55391756M,P.H. SECO BASICO 2C NORMAL,2025-10-271797755391756M
1,2025-10-27,17910,8452015,11971559J,FRUTOS SEC. S/CASCARA NUEZ,2025-10-271791011971559J
2,2025-10-27,17721,8452015,22454313B,FRUTOS SEC. S/CASCARA NUEZ,2025-10-271772122454313B
3,2025-10-27,17873,8452020,69353513A,FRUTOS SEC. S/CASCARA COCKTAIL,2025-10-271787369353513A
4,2025-10-27,17732,12650101,50171830K,P.H. SECO BASICO 2C NORMAL,2025-10-271773250171830K


In [120]:
print(junto["clave unica"].nunique())

188630


Qué productos hay en cada ticket

In [121]:
tickets_agrupados = junto.groupby("clave unica")["descripcion"].apply(list)


In [122]:
df2 = pd.DataFrame(tickets_agrupados)
df2.head(50)
df2.shape

(188630, 1)

Top 10 productos más comprados

In [123]:
Top_10= print(junto["descripcion"].value_counts().head(10))

descripcion
PLATANOS DE CANARIAS         41028
LECHE UHT ENTERA             40536
AGUA DE MESA                 37410
P.H. SECO BASICO 2C LARGO    35224
ZANAHORIAS                   34334
LECHE UHT SEMIDESNATADA      32294
PATATAS TODO USO             30062
ENTERO NATURAL               28397
JAMON DE CERDO               27903
PECHUGA DE PAVO              27783
Name: count, dtype: int64


In [124]:
print(junto["descripcion"].value_counts(ascending=True).head(50))

descripcion
RULOS DE CABRA RECIEN CORTADO            1
VERDURAS XXX                             1
CONTROL PESO BOTELLA                     1
NIÑOS ALTA SPRAY                         1
ACCESORIOS CAB.INFANT.DIADEMA            1
RETOCA RAICES CAOBAS-ROJIZOS             1
ACCESORIOS PARA LOTES                    1
INFANTIL                                 1
COMPR. PAN  SALUD Y CEREALES             1
COCKTAIL CON CASCARA                     1
AVES ESPECIALES                          1
CAMPAÑAS REGIONALES                      1
DENTIFRICOS                              1
CROQUETAS                                1
COMPR.  PAN TRADICIONAL                  1
COMPR. PAN BLANCO                        1
VASOS Y BIBERONES OTROS                  1
OTROS PESCADOS PLANOS                    1
UTILES CABELLO INFANT.PEINES             1
SAL OTROS USOS                           1
PROTECCION BAJA LOCION                   1
ESPECIAS                                 1
PROTECCION BAJA ACEITE                   1

In [125]:
import plotly.express as px

df_top10 = junto["descripcion"].value_counts().head(15).reset_index()
df_top10.columns = ["producto", "cantidad"]

fig = px.bar(
    df_top10,
    x="cantidad",
    y="producto",
    orientation="h",
    title="Cantidad comprada por producto (top 10 productos)",
    labels={"cantidad": "Cantidad", "producto": "Producto"},
    color_discrete_sequence=["red"]
)

fig.show()

Ver cuantos productos se compran por día

In [126]:
df_cantidad = junto.copy()

In [127]:
df_cantidad["cantidad"]= 1
dias_agrupados = df_cantidad.groupby("dia")["cantidad"].sum()

In [128]:
df_agrupado2 = pd.DataFrame(dias_agrupados)

In [129]:
df_agrupado2.head(50)
df_agrupado2.columns

Index(['cantidad'], dtype='str')

In [130]:
fig = px.line(
    df_agrupado2,
    x=df_agrupado2.index,
    y="cantidad",
    title="Análisis de ventas por día",
    labels={"dia": "Fecha", "cantidad": "Cantidad"},
    color_discrete_sequence=["red"]
)

fig.show()

In [131]:
productos_por_ticket = df_cantidad.groupby("clave unica")["cantidad"].sum()

In [132]:
df_productosportickets = pd.DataFrame(productos_por_ticket)

In [133]:
df_productosportickets= df_productosportickets.sort_values(by="cantidad",ascending=False)

In [134]:
df_productosportickets.head(10)

,cantidad
clave unica,
2025-08-021077733231253W,164
2025-10-041710840932066R,148
2025-08-121602882134436X,144
2025-08-061997721797096L,143
2025-08-021173379741132V,141
2025-09-201750612229808H,141
2025-09-231006424486487C,141
2025-10-081343980588461A,137
2025-09-101325280588461A,136


In [135]:
df_productosportickets.median()

cantidad    25.0
dtype: float64

In [136]:
df_cantidad["mes"]= df_cantidad["dia"].dt.month_name()
df_cantidad["mes_num"] = df_cantidad["dia"].dt.month

In [137]:
groupbymes = df_cantidad.groupby(["mes", "mes_num"])["cantidad"].sum().reset_index().sort_values("mes_num")

In [138]:
groupbymes

,mes,mes_num,cantidad
0,August,8,1123757
3,September,9,1349379
2,October,10,1388075
1,November,11,1275102


In [146]:
fig = px.bar(
    groupbymes,
    x='mes',
    y='cantidad',
    title='Ventas por mes',
    labels={'mes': 'Mes', 'cantidad': 'Ventas'},
    color='cantidad',
    color_continuous_scale='reds',
    text='cantidad'
)

fig.update_layout(
    xaxis_title='Mes',
    yaxis_title='Ventas',
    yaxis=dict(rangemode='tozero', showgrid=True, gridcolor='lightgrey'),
    xaxis=dict(tickmode='linear'),
    plot_bgcolor='white',
    coloraxis_colorbar=dict(title='Ventas'),
    legend_title_text='Ventas'
)

fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')

fig.show()

# Estandarizar para anonimizar

In [140]:
junto["id_cliente_enc"] = junto["id_cliente_enc"].str.strip().str.upper()

# Anonimizar

In [141]:
import hashlib

def anonimizar_dni(dni: str) -> str:
    return hashlib.sha256(dni.strip().upper().encode()).hexdigest()

junto["id_cliente_enc"]=junto["id_cliente_enc"].apply(anonimizar_dni)

junto.head()

,dia,num_ticket,cod_est,id_cliente_enc,descripcion,clave unica
0,2025-10-27,17977,12650101,2da19f2cfe3eb55bd1b3f7a6509d1584feec319661b780...,P.H. SECO BASICO 2C NORMAL,2025-10-271797755391756M
1,2025-10-27,17910,8452015,ee5123588e4cf0b305f476bda833bb5a5f09996a3800c0...,FRUTOS SEC. S/CASCARA NUEZ,2025-10-271791011971559J
2,2025-10-27,17721,8452015,327bd88127045855d8427dd89469f61ff81ca17946ad79...,FRUTOS SEC. S/CASCARA NUEZ,2025-10-271772122454313B
3,2025-10-27,17873,8452020,5f75e19069c8262c0762a3ed42276b71a42dd90c9367dc...,FRUTOS SEC. S/CASCARA COCKTAIL,2025-10-271787369353513A
4,2025-10-27,17732,12650101,d9b0f0d42754bd1be8e24370d68b8c7718a0da5b7fecbb...,P.H. SECO BASICO 2C NORMAL,2025-10-271773250171830K


# Matriz

In [142]:
m = df.pivot_table(
    index='id_cliente_enc',
    columns='cod_est',
    values='num_ticket',
    aggfunc='count',
    fill_value=0
)

matriz = (m.reset_index().melt(
    id_vars='id_cliente_enc',
    var_name='cod_producto',
    value_name='cantidad'
    ).query('cantidad > 0').reset_index(drop=True)
)

matriz.head(10)

,id_cliente_enc,cod_producto,cantidad
0,10044204N,1012305,4
1,10056687Y,1012305,1
2,10059715K,1012305,1
3,10063760H,1012305,1
4,10082243D,1012305,1
5,10098691N,1012305,2
6,10101152N,1012305,1
7,10111259E,1012305,1
8,10115232Q,1012305,1
9,10118956Z,1012305,2


In [143]:
matriz.describe()

,cantidad
count,2.763945e+06
mean,1.858327e+00
std,1.816528e+00
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,2.000000e+00
max,1.110000e+02


Limpiar si hay algún producto que nadie haya comprado

In [144]:
matriz_limpia = matriz.loc[:, (matriz != 0).any(axis=0)]
matriz_limpia.shape

(2763945, 3)